In [18]:
import regex as re

In [ ]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [179]:
# coding=utf8
# the above tag defines encoding for this document and is for Python 2.x compatibility

import regex as re

regex = r" ?(\w+)"

test_str = ("Yes 123 gjdj  Yes")

matches = re.finditer(regex, test_str, re.MULTILINE)

for matchNum, match in enumerate(matches, start=1):
    
    print ("Match {matchNum} was found at {start}-{end}: {match}".format(matchNum = matchNum, start = match.start(), end = match.end(), match = match.group()))
    
    for groupNum in range(0, len(match.groups())):
        groupNum = groupNum + 1
        
        print ("Group {groupNum} found at {start}-{end}: {group}".format(groupNum = groupNum, start = match.start(groupNum), end = match.end(groupNum), group = match.group(groupNum)))

# Note: for Python 2.7 compatibility, use ur"" to prefix the regex and u"" to prefix the test string and substitution.


Match 1 was found at 0-3: Yes
Group 1 found at 0-3: Yes
Match 2 was found at 3-7:  123
Group 1 found at 4-7: 123
Match 3 was found at 7-12:  gjdj
Group 1 found at 8-12: gjdj
Match 4 was found at 13-17:  Yes
Group 1 found at 14-17: Yes


In [181]:
match.groups()

('Yes',)

In [154]:
PAT.split('|')

["'(?:[sdmt]",
 'll',
 've',
 're)',
 ' ?\\p{L}+',
 ' ?\\p{N}+',
 ' ?[^\\s\\p{L}\\p{N}]+',
 '\\s+(?!\\S)',
 '\\s+']

In [229]:
re_pat = re.compile(PAT)
re_pat

regex.Regex("'(?:[sdmt]|ll|ve|re)| ?\\p{L}+| ?\\p{N}+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+", flags=regex.V0)

In [23]:
t = """
    def train_bpe(input_path: str,
                vocab_size: int,
                special_tokens: list[str]) -> tuple[dict[int, bytes], list[tuple[bytes, bytes]]]:
        
        vocab: dict[int, bytes] = {idx : bytes([idx]) for idx in range(256)} #initial vocab
        merges: list[tuple[bytes, bytes]] = []

        # ensure file exists
        if not os.path.exists(input_path):
            raise FileNotFoundError(f"File not found at {input_path}")
        
        # pretokenize text

        # get num of merges
        num_merges = vocab_size - 256
        
        return vocab, merges
    """

In [192]:
ts = re.finditer(PAT, t)
for i in ts:
    print(i.group())
    


   
 def
 train
_
bpe
(
input
_
path
:
 str
,

               
 vocab
_
size
:
 int
,

               
 special
_
tokens
:
 list
[
str
])
 ->
 tuple
[
dict
[
int
,
 bytes
],
 list
[
tuple
[
bytes
,
 bytes
]]]:


       
 vocab
:
 dict
[
int
,
 bytes
]
 =
 {
idx
 :
 bytes
([
idx
])
 for
 idx
 in
 range
(
256
)}
 #
initial
 vocab

       
 merges
:
 list
[
tuple
[
bytes
,
 bytes
]]
 =
 []


       
 #
 ensure
 file
 exists

       
 if
 not
 os
.
path
.
exists
(
input
_
path
):

           
 raise
 FileNotFoundError
(
f
"
File
 not
 found
 at
 {
input
_
path
}")


       
 #
 pretokenize
 text


       
 #
 get
 num
 of
 merges

       
 num
_
merges
 =
 vocab
_
size
 -
 256


       
 return
 vocab
,
 merges

    


## mismatch test

In [1]:
import numpy as np
import pickle

In [8]:
p = "tests/_snapshots/test_train_bpe_special_tokens.pkl"

with open(p, 'rb') as f:
    data = pickle.load(f)

In [30]:
p = "v1.pkl"

with open(p, 'rb') as f:
    data = pickle.load(f)

In [31]:
p = "v2.pkl"

with open(p, 'rb') as f:
    data2 = pickle.load(f)

In [32]:
len(data["merges"]) == len(data2['merges'])

True

In [34]:
for i, (a, b) in enumerate(zip(data["merges"], data2["merges"])):
    if a != b:
        print(f"Mismatch at idx{i}: ground_truth = {a}, my_merges = {b}")

In [35]:
for i, (a, b) in enumerate(zip(data["merges"], data2["merges"])):
    if a != b:
        print(f"Mismatch at idx{i}: ground_truth = {a}, my_merges = {b}")

In [8]:
for i, (a, b) in enumerate(zip(data["merges"], data2["merges"])):
    if a != b:
        print(f"Mismatch at idx{i}: ground_truth = {a}, my_merges = {b}")
    #else:
        #print(f"No Mismatch at idx{i}: ground_truth = {a}, my_merges = {b}")

In [25]:
from collections import Counter, defaultdict

In [ ]:
def merge1(ids: list[bytes],  
          pair: tuple[bytes, bytes],
          num_occurence: int, 
          local_count_delta: Counter[tuple[bytes, bytes]],
          delta_pos) -> tuple[list[bytes], Counter[tuple[bytes, bytes]], dict[tuple[bytes, bytes], int]]:
    
    A, B = pair[0], pair[1]
    C = A + B

    idx = merges_done = 0
    

    while idx < len(ids):
        if idx < len(ids) - 1 and ids[idx] == A and ids[idx + 1] == B:
            merges_done += 1

            if idx > 0:
                local_count_delta[(ids[idx - 1], ids[idx])] -= 1
                local_count_delta[(ids[idx - 1], C)] += 1
                delta_pos[(ids[idx - 1], ids[idx])] -= 1
                delta_pos[(ids[idx - 1], C)] += 1
                
                
            
            ids[idx] = C
            del ids[idx + 1]

            if idx + 1 < len(ids):
                local_count_delta[(B, ids[idx + 1])] -= 1
                local_count_delta[(C, ids[idx + 1])] += 1
                delta_pos[(ids[idx - 1], ids[idx])] -= 1
                delta_pos[(ids[idx - 1], C)] += 1
        
        idx += 1
        if merges_done >= num_occurence:
            break
    
    return ids, local_count_delta, delta_pos

In [ ]:
def merge(ids: list[bytes], pair: tuple[bytes, bytes], local_delta: Counter[tuple[bytes, bytes]]) -> tuple[list[bytes], Counter[tuple[bytes, bytes]]]:
    A, B = pair[0], pair[1]  # pair is e.g (b'h', b'e')
    new_val = A + B  # new_val wil be b'he'
    
    # two pointer in place merge
    write = read = 0
    while read < len(ids):
        if read < len(ids) - 1 and ids[read] == A and ids[read + 1] == B:
            # pair found. record the change for this list of ids by removing that pair from the count
            local_delta[pair] -= 1

            # remove pair to the left of found pair
            if write > 0:
                local_delta[(ids[write - 1], ids[read])] -= 1
                local_delta[(ids[write - 1], new_val)] += 1
            
            ids[write] = new_val # write new_val into list of bytes
            read += 2 # skip the next element

            if read < len(ids):
                local_delta[(ids[read - 1], ids[read])] -= 1
                local_delta[(new_val, ids[read])] += 1
        else:
            ids[write] = ids[read]
            read += 1
        write += 1

    del ids[write:]
   
    return ids, local_delta

### something else

In [1]:
import regex as re
text = open("tests/fixtures/tinystories_sample_5M.txt", "r", encoding="utf-8").read()

# split corpus on special tokens
special_pat = "|".join(map(re.escape, ['<|endoftext|>']))

parts = re.split(special_pat, text)

text = "".join(parts)

In [3]:
import os
import time
from new_bpe import RegexTokenizer

# create a directory for models, so we don't pollute the current directory
os.makedirs("models", exist_ok=True)

t0 = time.time()

# construct the Tokenizer object and kick off verbose training
tokenizer = RegexTokenizer(special_tokens={'<|endoftext|>': 999})
tokenizer.train(text, 1000, verbose=True)
# writes two files in the models directory: name.model, and name.vocab
#prefix = os.path.join("models", "regex")
#tokenizer.save(prefix)
t1 = time.time()

print(f"Training took {t1 - t0:.2f} seconds")

merge 1/743: (104, 101) -> 256 (b'he') had 148935 occurrences
merge 2/743: (32, 116) -> 257 (b' t') had 148857 occurrences
merge 3/743: (32, 97) -> 258 (b' a') had 111510 occurrences
merge 4/743: (32, 115) -> 259 (b' s') had 76372 occurrences
merge 5/743: (32, 119) -> 260 (b' w') had 74350 occurrences
merge 6/743: (110, 100) -> 261 (b'nd') had 67904 occurrences
merge 7/743: (257, 256) -> 262 (b' the') had 67683 occurrences
merge 8/743: (101, 100) -> 263 (b'ed') had 58392 occurrences
merge 9/743: (32, 98) -> 264 (b' b') had 52390 occurrences
merge 10/743: (257, 111) -> 265 (b' to') had 49394 occurrences
merge 11/743: (258, 261) -> 266 (b' and') had 45801 occurrences
merge 12/743: (32, 104) -> 267 (b' h') had 42014 occurrences
merge 13/743: (32, 102) -> 268 (b' f') had 39838 occurrences
merge 14/743: (105, 110) -> 269 (b'in') had 38758 occurrences
merge 15/743: (260, 97) -> 270 (b' wa') had 38433 occurrences
merge 16/743: (32, 84) -> 271 (b' T') had 38135 occurrences
merge 17/743: (105, 

In [17]:
(b' s') < (b'i')

True

In [53]:
stats = {(b' sa'): 200, (b'im'): 200}
pair = max(stats, key=lambda x: (stats[x], x))

In [54]:
pair

b'im'

In [31]:
new = [tokenizer.vocab[v] for k, v in tokenizer.merges.items()]

In [10]:
new2 = [(tokenizer.vocab[k[0]], tokenizer.vocab[k[1]]) for k in tokenizer.merges.keys()]

In [16]:
len(new2) == len(data['merges'])

True

In [12]:
new2[100]

(b'a', b'll')

In [13]:
data['merges'][100]

(b'a', b'll')

In [15]:
for i, (a, b) in enumerate(zip(data["merges"], new2)):
    if a != b:
        print(f"Mismatch at idx{i}: ground_truth = {a}, my_merges = {b}")

Mismatch at idx34: ground_truth = (b'i', b'm'), my_merges = (b' s', b'a')
Mismatch at idx35: ground_truth = (b' s', b'a'), my_merges = (b'i', b'm')
Mismatch at idx81: ground_truth = (b' h', b'is'), my_merges = (b' S', b'he')
Mismatch at idx82: ground_truth = (b' S', b'he'), my_merges = (b' h', b'is')
Mismatch at idx108: ground_truth = (b' w', b'e'), my_merges = (b' ha', b'd')
Mismatch at idx109: ground_truth = (b' ha', b'd'), my_merges = (b' w', b'e')
Mismatch at idx233: ground_truth = (b'a', b'x'), my_merges = (b' g', b'o')
Mismatch at idx234: ground_truth = (b' g', b'o'), my_merges = (b'a', b'x')
Mismatch at idx242: ground_truth = (b' tre', b'e'), my_merges = (b'\n', b'\n')
Mismatch at idx243: ground_truth = (b' c', b'l'), my_merges = (b' tre', b'e')
Mismatch at idx244: ground_truth = (b' lo', b'ved'), my_merges = (b' c', b'l')
Mismatch at idx245: ground_truth = (b'ot', b'her'), my_merges = (b' lo', b'ved')
Mismatch at idx246: ground_truth = (b' b', b'ack'), my_merges = (b'ot', b'her

In [28]:
tokenizer.vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'